# 🏛️ Contract Analysis AI — Local CPU Fast Training

This notebook is optimized for **fast local CPU training** on your computer. 
It subsamples the large Hugging Face contract dataset and trains a **DistilBERT** model for **1 epoch** to test and verify the local development workspace in under 2 minutes.

### Steps:
1. 📥 Load and subsample the dataset (100 training, 20 test samples).
2. 🔤 Clean text and encode categories.
3. 🧠 Load the base DistilBERT model.
4. 🏋️ Train on CPU for 1 epoch.
5. 💾 Save the model weights directly to the `models/` directory for FastAPI server use.

In [ ]:
import os, json, re, warnings
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate
import numpy as np
warnings.filterwarnings('ignore')

print("✅ Core libraries loaded. PyTorch version:", torch.__version__)
print("CUDA GPU available:", torch.cuda.is_available())

In [ ]:
DATASET_NAME = 'alisha4walunj/quad_ledgar_merged_dataset'
print(f"𰲃 Loading dataset '{DATASET_NAME}' from HuggingFace Hub...")
raw = load_dataset(DATASET_NAME)
ds = raw['train']

# Subsample dataset for ultra-fast local CPU training
ds = ds.shuffle(seed=42).select(range(120))
print(f"Total subsampled examples: {len(ds)}")

In [ ]:
# Clean text
def clean_text(text):
    if not isinstance(text, str): text = str(text)
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'[^\x20-\x7E]+', ' ', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip().lower()

print("✨ Cleaning clause text...")
ds = ds.map(lambda b: {'text': [clean_text(t) for t in b['text']]}, batched=True)

# Map labels to IDs
unique_labels = sorted(set(ds['label']))
label2id = {lbl: i for i, lbl in enumerate(unique_labels)}
id2label = {i: lbl for lbl, i in label2id.items()}

print(f"Classes detected ({len(unique_labels)}):", unique_labels[:5], "...")
ds = ds.map(lambda b: {'labels': [label2id[l] for l in b['label']]}, batched=True)

# Split into Train and Test
splits = ds.train_test_split(test_size=20, seed=42) # 100 train, 20 test
print(f"Train size: {len(splits['train'])} | Test size: {len(splits['test'])}")

In [ ]:
print("🔤 Initializing tokenizer...")
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=512)

tokenized = splits.map(tokenize_fn, batched=True)

# Remove raw text and label columns that the Trainer cannot convert to tensors
keep_cols = {'input_ids', 'attention_mask', 'labels'}
for split in tokenized:
    remove_cols = [c for c in tokenized[split].column_names if c not in keep_cols]
    tokenized[split] = tokenized[split].remove_columns(remove_cols)

tokenized.set_format('torch')
print("Tokenization complete!")
print("Columns left for training:", tokenized['train'].column_names)

In [ ]:
print("🧠 Initializing DistilBERT base model...")
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Define metrics
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)['accuracy']
    f1 = f1_metric.compute(predictions=preds, references=labels, average='weighted')['f1']
    return {'accuracy': acc, 'f1': f1}

print("✅ Model loaded and ready!")

In [ ]:
# Set save directory to models/contract_classifier/
output_dir = '../models/contract_classifier'
os.makedirs(output_dir, exist_ok=True)

print("🏋️ Configuring Training Arguments and launching training...")
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='none',
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
print("💾 Saving final model weights and tokenizer...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

# Save label map separately in the models folder for FastAPI
label_map_path = '../models/label_map.json'
with open(label_map_path, 'w') as f:
    json.dump({str(k): v for k, v in id2label.items()}, f, indent=2)
    
print("🎉 SUCCESS: Model fully trained and saved to: models/contract_classifier/")
print("💡 Now run the server with: uv run uvicorn app.main:app --reload")